# Kvasir-SEG 용종 마스크 검수

공개 내시경 영상에서 경량 모델의 후보 마스크를 검수하는 Colab 노트북입니다. 기존 모델과 불확실성 점수를 사용해 순서를 제시하며, 사람이 승인·수정·보류를 결정합니다.

**GPU 선택 → 모두 실행 → 마지막 셀의 주소 열기 → 검수 → 결과 내보내기**

Colab에서 외부 주소는 임시 인증으로 보호합니다. 공개 연구 데이터만 사용하세요. Kvasir-SEG는 연구·교육용이며, 상업적 사용에는 사전 서면 허가가 필요합니다. [공식 이용 조건](https://datasets.simula.no/kvasir-seg/) · [Jha et al., 2020](https://doi.org/10.1007/978-3-030-37734-2_37)


## 01 · 실행 설정

기본 설정에서 Colab의 **런타임 → 런타임 유형 변경 → GPU → 모두 실행**을 사용합니다. 이전 기준 모델이 있으면 자동으로 재사용합니다. 파일을 직접 고르려면 `UPLOAD_CHECKPOINT=True`를 설정합니다. 합성 데이터 실행은 `PROFILE='smoke'`입니다.

In [ ]:
# @title 실행 설정
from pathlib import Path
import importlib.metadata as metadata
import os
import sys
import subprocess

PROFILE = "research"  # @param ["research", "smoke"]
CHECKPOINT_PATH = ""  # @param {type:"string"}
UPLOAD_CHECKPOINT = False  # @param {type:"boolean"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
WORK_ROOT_OVERRIDE = ""  # @param {type:"string"}
LOCAL_ARCHIVE = ""  # @param {type:"string"}
BATCH_SIZE = 8  # @param {type:"integer"}
NUM_WORKERS = 2  # @param {type:"integer"}

GRADIO_VERSION = "6.32.0"
IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")
SMOKE = PROFILE == "smoke"
if PROFILE not in {"research", "smoke"} or BATCH_SIZE < 2 or NUM_WORKERS < 0:
    raise ValueError("PROFILE, BATCH_SIZE (2 이상), NUM_WORKERS (0 이상)를 확인하세요.")

packages = {
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
    "matplotlib": "matplotlib", "Pillow": "Pillow",
}
requirements = []
for distribution, specification in packages.items():
    try:
        metadata.version(distribution)
    except metadata.PackageNotFoundError:
        requirements.append(specification)
try:
    gradio_ready = metadata.version("gradio") == GRADIO_VERSION
except metadata.PackageNotFoundError:
    gradio_ready = False
if not gradio_ready:
    if "gradio" in sys.modules:
        raise RuntimeError("Gradio가 이미 로드되었습니다. Colab 런타임을 다시 시작하고 위에서 실행하세요.")
    requirements.append(f"gradio=={GRADIO_VERSION}")
if requirements:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *requirements], check=True)

import hashlib
import io
import json
import math
import platform
import pickle
import random
import secrets
import shutil
import sqlite3
import tempfile
import time
import urllib.request
import warnings
import zipfile
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from threading import RLock

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from scipy import ndimage
try:
    import torch
    import torchvision
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
    from torchvision.models.segmentation import (
        LRASPP_MobileNet_V3_Large_Weights, lraspp_mobilenet_v3_large,
    )
except (ImportError, OSError, RuntimeError) as error:
    raise RuntimeError("Colab 기본 런타임의 torch와 torchvision 조합을 확인하세요.") from error

if gr.__version__ != GRADIO_VERSION:
    raise RuntimeError(f"Gradio가 {gr.__version__}으로 로드되었습니다. 런타임을 재시작하세요.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
if MOUNT_DRIVE:
    if not IN_COLAB:
        raise ValueError("MOUNT_DRIVE는 Colab에서 사용할 수 있습니다.")
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = Path(WORK_ROOT_OVERRIDE or (
    "/content/drive/MyDrive/kvasir_review" if MOUNT_DRIVE else
    "/content/kvasir_review" if IN_COLAB else "kvasir_review_work"
)).resolve()
CACHE = Path("/content/polyp_audit_cache") if IN_COLAB else ROOT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(CACHE / "torch_hub"))
config_versions = {name: metadata.version(name) for name in ["torch", "torchvision", "gradio", *packages]}
ENVIRONMENT = {"python": platform.python_version(), "packages": config_versions,
               "device": DEVICE.type, "gpu": torch.cuda.get_device_name() if DEVICE.type == "cuda" else None}

CONFIG = {
    "protocol": "review-1.0", "profile": PROFILE, "seed": 2026,
    "side": 64 if SMOKE else 256, "epochs": 1 if SMOKE else 12,
    "batch_size": min(BATCH_SIZE, 4) if SMOKE else BATCH_SIZE,
    "workers": 0 if SMOKE else NUM_WORKERS, "amp": DEVICE.type == "cuda" and not SMOKE,
    "learning_rate": 5e-4, "backbone_learning_rate": 1e-4, "weight_decay": 1e-4,
    "probability_threshold": 0.50, "group_dhash_distance": 6, "group_rgb_mae": 0.08,
    "split_fractions": {"train": 0.70, "val_select": 0.10, "val_audit": 0.10, "test": 0.10},
    "tta_flips": [[], [-1], [-2], [-2, -1]], "entropy_top_fraction": 0.05,
    "pretrained": not SMOKE, "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],
    "review_budgets": [0.1, 0.2, 0.3], "random_repeats": 500, "injection_repeats": 12,
    "injection_fraction": 0.50, "injection_types": ["shift", "erosion", "dilation", "local_missing"],
    "contour_max_distance": 8, "contour_top_fraction": 0.25, "contour_weight": 0.65,
}
RESUME = True

# 기존 실험의 시드와 저장 유틸리티
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
                         encoding="utf-8")
    temporary.replace(path)

def save_csv(frame, filename):
    target = RUN / filename
    temporary = target.with_suffix(".csv.part")
    frame.to_csv(temporary, index=False)
    temporary.replace(target)

seed_everything(CONFIG['seed'])

## 02 · 데이터와 분할

공식 Simula 데이터와 원저자의 Kaggle v3 배포 경로를 사용합니다. 같은 이미지와 근접 중복은 한 분할에 두고, 평가용으로 남겨둔 기존 테스트셋은 이번 검수에 사용하지 않습니다.

In [ ]:
OFFICIAL_URLS = [
    "https://datasets.simula.no/downloads/kvasir-seg.zip",
    "https://datasets.simula.no/kvasir-seg/Kvasir-SEG.zip",
    "https://www.kaggle.com/api/v1/datasets/download/debeshjha1/kvasirseg?datasetVersionNumber=3",
]

OFFICIAL_PAGE = "https://datasets.simula.no/kvasir-seg/"

PAPER = "https://doi.org/10.1007/978-3-030-37734-2_37"

EXPECTED_ARCHIVE_SHA256 = "03b30e21d584e04facf49397a2576738fd626815771afbbf788f74a7153478f7"

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}

def pair_files(root, expected):
    def index(folder):
        result = {}
        for path in sorted(folder.iterdir()):
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES:
                if path.stem in result:
                    raise ValueError(f"중복 샘플 ID: {path.stem}")
                result[path.stem] = path
        return result

    images, masks = (index(root / directory) for directory in ("images", "masks"))
    if set(images) != set(masks):
        raise ValueError(f"이미지/마스크 ID 불일치: {len(set(images) ^ set(masks))}개")
    if len(images) != expected:
        raise ValueError(f"예상 {expected}쌍, 실제 {len(images)}쌍")
    return [(key, images[key], masks[key]) for key in sorted(images)]

def zip_root(archive, destination):
    with zipfile.ZipFile(archive) as package:
        members = package.infolist()
        if len(members) > 5000 or sum(item.file_size for item in members) > 500_000_000:
            raise ValueError("Kvasir-SEG ZIP의 예상 크기를 초과합니다.")
        for member in members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError(f"안전하지 않은 ZIP 경로: {member.filename}")
        package.extractall(destination)
    roots = [folder for folder in destination.rglob("images")
             if folder.is_dir() and (folder.parent / "masks").is_dir()
             and "__MACOSX" not in folder.parts]
    if len(roots) != 1:
        raise ValueError("images/와 masks/가 있는 데이터 폴더를 하나 찾을 수 있어야 합니다.")
    return roots[0].parent

def get_official_data():
    default_archive = CACHE / "kvasir-seg.zip"
    uploaded = Path(LOCAL_ARCHIVE).expanduser() if LOCAL_ARCHIVE else None
    if uploaded is not None and not uploaded.is_file():
        raise FileNotFoundError(f"LOCAL_ARCHIVE의 공식 ZIP을 찾을 수 없습니다: {uploaded}")
    sources = [(uploaded, "user_supplied_official_archive")] if uploaded else []
    if not uploaded:
        if default_archive.exists():
            sources.append((default_archive, "cached_official_archive"))
        sources.extend((default_archive, url) for url in OFFICIAL_URLS)
    cached_provenance = CACHE / "kvasir-seg.source.json"
    errors = []
    for archive, source in sources:
        attempts = 2 if source.startswith("https://") else 1
        for attempt in range(attempts):
            partial = CACHE / "kvasir-seg.zip.part"
            staging = CACHE / "extracting"
            try:
                if source.startswith("https://"):
                    request = urllib.request.Request(source, headers={"User-Agent": "KvasirSEG-Research/1.0"})
                    with urllib.request.urlopen(request, timeout=35) as response, partial.open("wb") as stream:
                        shutil.copyfileobj(response, stream, 1024 * 1024)
                    file_to_verify = partial
                else:
                    file_to_verify = archive
                fingerprint = sha256_file(file_to_verify)
                if source == OFFICIAL_URLS[0] and fingerprint != EXPECTED_ARCHIVE_SHA256:
                    raise ValueError(f"공개 기록과 기본 ZIP의 SHA-256 불일치: {fingerprint}")
                if fingerprint != EXPECTED_ARCHIVE_SHA256:
                    print(f"기본 ZIP과 다른 SHA-256 ({source}): {fingerprint}. 구조 검사를 계속합니다.")
                extracted = CACHE / f"kvasir_{fingerprint}"
                if not extracted.exists():
                    shutil.rmtree(staging, ignore_errors=True)
                    staging.mkdir()
                    origin = zip_root(file_to_verify, staging)
                    pair_files(origin, expected=1000)
                    origin.replace(extracted)
                    shutil.rmtree(staging, ignore_errors=True)
                pairs = pair_files(extracted, expected=1000)
                provenance = {"source": source, "archive_sha256": fingerprint,
                              "matches_simula_primary_sha256": fingerprint == EXPECTED_ARCHIVE_SHA256}
                if file_to_verify == partial:
                    partial.replace(default_archive)
                    write_json(cached_provenance, provenance)
                elif source == "cached_official_archive" and cached_provenance.is_file():
                    recorded = json.loads(cached_provenance.read_text(encoding="utf-8"))
                    if recorded.get("archive_sha256") == fingerprint:
                        provenance["source"] = recorded["source"]
                return pairs, provenance
            except (OSError, ValueError, RuntimeError, zipfile.BadZipFile) as error:
                errors.append(f"{source} ({attempt + 1}/{attempts}): {error}")
                partial.unlink(missing_ok=True)
                shutil.rmtree(staging, ignore_errors=True)
                if attempt + 1 < attempts:
                    time.sleep(2)
    raise RuntimeError("공식 Kvasir-SEG ZIP 준비 실패. Simula 또는 저자의 Kaggle 배포본 ZIP을 내려받아 Colab에 "
                       "업로드한 뒤 LOCAL_ARCHIVE를 지정하고 다시 실행하세요.\n" + "\n".join(errors))

def smoke_fixtures():
    root = CACHE / "smoke_fixture_v1"
    for name in ("images", "masks"):
        (root / name).mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(CONFIG["seed"])
    y, x = np.mgrid[:72, :88]
    for index in range(48):
        height = 72 if index % 2 else 60
        cx, cy = rng.integers([23, 20], [64, 41])
        rx, ry = rng.integers([4, 4], [23, 18])
        mask = ((x - cx) / rx) ** 2 + ((y - cy) / ry) ** 2 < 1
        if index % 3 == 0:
            mask |= ((x - cx - 0.7 * rx) ** 2 + (y - cy) ** 2 < (ry * 0.65) ** 2)
        rgb = rng.integers(20, 180, (72, 88, 3), dtype=np.uint8)
        rgb[mask, 0] = 225
        key = f"fixture_{index:03d}.png"
        Image.fromarray(rgb[:height]).save(root / "images" / key)
        Image.fromarray(mask[:height].astype("uint8") * 255).save(root / "masks" / key)
    return pair_files(root, 48), {"source": "synthetic_smoke_fixtures", "archive_sha256": None}

def image_fingerprint(path):
    with Image.open(path) as image:
        image = image.convert("RGB")
        digest = hashlib.sha256(f"{image.size}".encode() + image.tobytes()).hexdigest()
        thumb = np.asarray(image.resize((32, 32), Image.Resampling.BILINEAR), dtype=np.int16)
        gray = np.asarray(image.convert("L").resize((9, 9), Image.Resampling.BILINEAR))
    horizontal = gray[:8, 1:] > gray[:8, :-1]
    vertical = gray[1:, :8] > gray[:-1, :8]
    dhash = int.from_bytes(np.packbits(np.r_[horizontal.ravel(), vertical.ravel()]).tobytes(), "big")
    return digest, dhash, thumb

def grouped_split(pairs):
    fingerprints = [image_fingerprint(image) for _, image, _ in pairs]
    parents = list(range(len(pairs)))

    def root(index):
        while parents[index] != index:
            parents[index] = parents[parents[index]]
            index = parents[index]
        return index

    edges = []
    for i, (digest, dhash, thumb) in enumerate(fingerprints):
        for j in range(i):
            other_digest, other_hash, other_thumb = fingerprints[j]
            distance = (dhash ^ other_hash).bit_count()
            if digest != other_digest and distance > CONFIG["group_dhash_distance"]:
                continue
            mae = float(np.abs(thumb - other_thumb).mean() / 255)
            if digest == other_digest or mae <= CONFIG["group_rgb_mae"]:
                a, b = root(i), root(j)
                parents[max(a, b)] = min(a, b)
                edges.append((pairs[j][0], pairs[i][0], distance, mae, digest == other_digest))
    groups = defaultdict(list)
    for index, (key, _, _) in enumerate(pairs):
        groups[pairs[root(index)][0]].append(key)

    fractions = CONFIG["split_fractions"]
    names = list(fractions)
    targets = np.array(list(fractions.values())) * len(pairs)
    counts = np.zeros(len(names), dtype=int)
    rng = np.random.default_rng(CONFIG["seed"])
    group_ids = rng.permutation(sorted(groups)).tolist()
    group_ids.sort(key=lambda group: -len(groups[group]))  # Stable order after seeded permutation.
    rows = []
    for group in group_ids:
        slot = int(np.argmax(targets - counts))
        rows.extend((key, group, names[slot]) for key in groups[group])
        counts[slot] += len(groups[group])

    split = pd.DataFrame(rows, columns=["id", "group_id", "split"]).sort_values("id")
    split["decoded_rgb_sha256"] = [item[0] for item in fingerprints]
    split = split.reset_index(drop=True)
    assert split.id.is_unique and len(split) == len(pairs)
    assert set(split.split) == set(names) and split.groupby("group_id").split.nunique().max() == 1
    assert split.groupby("decoded_rgb_sha256").split.nunique().max() == 1
    edge_table = pd.DataFrame(edges, columns=["id_a", "id_b", "dhash_distance", "rgb_mae", "exact"])
    return split, edge_table

PAIRS, DATA_INFO = smoke_fixtures() if SMOKE else get_official_data()
SPLIT, DUPLICATE_EDGES = grouped_split(PAIRS)
SPLIT_SHA = hashlib.sha256(SPLIT.to_csv(index=False).encode()).hexdigest()
SPLIT_IDS = {name: SPLIT.loc[SPLIT.split.eq(name), "id"].tolist()
             for name in CONFIG["split_fractions"]}
PAIR_BY_ID = {key: (key, image, mask) for key, image, mask in PAIRS}
MEAN = np.asarray(CONFIG["mean"], dtype=np.float32)
STD = np.asarray(CONFIG["std"], dtype=np.float32)
training_signature = {"config": CONFIG, "split_sha256": SPLIT_SHA,
                      "archive_sha256": DATA_INFO["archive_sha256"],
                      "device": DEVICE.type, "versions": config_versions}
RUN_ID = hashlib.sha256(json.dumps(training_signature, sort_keys=True).encode()).hexdigest()[:12]
RUN = ROOT / "training" / RUN_ID
(RUN / "checkpoints").mkdir(parents=True, exist_ok=True)

# A companion run_setup makes checkpoint provenance verifiable.
training_manifest = {"run_id": RUN_ID, **training_signature}
write_json(RUN / "run_setup.json", training_manifest)
save_csv(SPLIT, "split_manifest.csv")
print(f"{DATA_INFO['source']} · {len(PAIRS)}쌍 · 분할 {SPLIT.split.value_counts().to_dict()}")

## 03 · 기준 모델 준비

기존 LR-ASPP MobileNetV3-Large를 사용합니다. 호환 체크포인트가 없으면 기존 학습 방식을 적용합니다. 이전 실행의 가중치, 메타데이터 검증 상태와 선택 epoch를 표시합니다.

In [ ]:
def letterbox(image, mask, side):
    width, height = image.size
    ratio = min(side / width, side / height)
    w, h = max(1, round(width * ratio)), max(1, round(height * ratio))
    x, y = (side - w) // 2, (side - h) // 2
    rgb = np.full((side, side, 3), [124, 116, 104], dtype=np.uint8)
    target = np.zeros((side, side), dtype=bool)
    valid = np.zeros_like(target)
    roi = np.s_[y:y + h, x:x + w]
    rgb[roi] = np.asarray(image.resize((w, h), Image.Resampling.BILINEAR))
    target[roi] = np.asarray(mask.resize((w, h), Image.Resampling.NEAREST)) >= 128
    valid[roi] = True
    return rgb, target, valid

def load_pair(key, image_path, mask_path):
    try:
        with Image.open(image_path) as image, Image.open(mask_path) as mask:
            if image.size != mask.size:
                raise ValueError(f"이미지 크기 {image.size}, 마스크 크기 {mask.size}")
            return letterbox(image.convert("RGB"), mask.convert("L"), CONFIG["side"])
    except (OSError, ValueError) as error:
        raise RuntimeError(f"{key} 전처리 실패: {error}") from error

class PolypDataset(Dataset):
    def __init__(self, keys, augment=False, epoch=0):
        self.pairs = [PAIR_BY_ID[key] for key in keys]
        self.augment = augment
        self.epoch = epoch

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        key, image_path, mask_path = self.pairs[index]
        rgb, target, valid = load_pair(key, image_path, mask_path)
        if self.augment:
            rng = np.random.default_rng([CONFIG["seed"], self.epoch, index])
            for axis in (0, 1):
                if rng.random() < 0.5:
                    rgb, target, valid = (np.flip(item, axis) for item in (rgb, target, valid))
            adjusted = np.clip(rgb.astype(np.float32) * rng.uniform(0.90, 1.10), 0, 255)
            rgb = np.where(valid[..., None], adjusted, rgb)
        normalized = (rgb.astype(np.float32) / 255.0 - MEAN) / STD
        return (torch.from_numpy(normalized.transpose(2, 0, 1).copy()),
                torch.from_numpy(target[None].copy()), torch.from_numpy(valid[None].copy()), key)

def loader(split, epoch=0):
    training = split == "train"
    generator = torch.Generator().manual_seed(CONFIG["seed"] + epoch)
    return DataLoader(PolypDataset(SPLIT_IDS[split], training, epoch), batch_size=CONFIG["batch_size"],
                      shuffle=training, num_workers=CONFIG["workers"], generator=generator,
                      pin_memory=DEVICE.type == "cuda")

def mask_geometry(target, valid):
    target = target & valid
    # Four-connected exposed grid edges give a perimeter proxy in input pixels.
    padded = np.pad(target, 1).astype(np.int16)
    perimeter = np.abs(np.diff(padded, axis=0)).sum() + np.abs(np.diff(padded, axis=1)).sum()
    area = int(target.sum())
    components = int(ndimage.label(target)[1])
    return {
        "area_fraction": area / int(valid.sum()), "area_pixels": area,
        "boundary_complexity": float(perimeter ** 2 / (4 * np.pi * area)) if area else np.nan,
        "components": components,
        "touches_valid_edge": bool((target & ~ndimage.binary_erosion(valid)).any()),
    }

def build_model(pretrained=False):
    weights = LRASPP_MobileNet_V3_Large_Weights.COCO_WITH_VOC_LABELS_V1 if pretrained else None
    try:
        model = lraspp_mobilenet_v3_large(weights=weights, weights_backbone=None)
    except (OSError, RuntimeError) as error:
        raise RuntimeError("TorchVision 사전학습 가중치/API 초기화 실패. 연결 상태와 "
                           "torch/torchvision 버전 조합을 확인하고 다시 실행하세요.") from error
    for name in ("low_classifier", "high_classifier"):
        old = getattr(model.classifier, name)
        setattr(model.classifier, name, nn.Conv2d(old.in_channels, 1, kernel_size=1))
    return model

def save_checkpoint(path, payload):
    temp = path.with_suffix(".pt.part")
    torch.save(payload, temp)
    temp.replace(path)

def initialization():
    path = RUN / "checkpoints" / "shared_initialization.pt"
    if not path.exists() or not RESUME:
        seed_everything(CONFIG["seed"])
        model = build_model(CONFIG["pretrained"]).eval()
        with torch.inference_mode():
            side = CONFIG["side"]
            output = model.to(DEVICE)(torch.zeros((1, 3, side, side), device=DEVICE))["out"]
        assert output.shape == (1, 1, side, side) and torch.isfinite(output).all()
        save_checkpoint(path, model.cpu().state_dict())
    return path

def metric_rows(predictions, targets, valid_pixels, keys, detailed=False):
    def ratio(numerator, denominator, empty):
        return numerator / denominator if denominator else float(empty)

    def boundary_f1(prediction, target, valid):
        a = prediction & ~ndimage.binary_erosion(prediction)
        b = target & ~ndimage.binary_erosion(target)
        if not a.any() or not b.any():
            return float(a.any() == b.any())
        tolerance = CONFIG["boundary_tolerance_px"]
        match_a = (a & valid & (ndimage.distance_transform_edt(~b) <= tolerance)).sum() / a.sum()
        match_b = (b & valid & (ndimage.distance_transform_edt(~a) <= tolerance)).sum() / b.sum()
        return ratio(2 * match_a * match_b, match_a + match_b, False)

    rows = []
    for key, pred, target, valid in zip(keys, predictions, targets, valid_pixels, strict=True):
        pred, target = pred & valid, target & valid
        tp = int((pred & target).sum())
        fp = int((pred & ~target).sum())
        fn = int((~pred & target).sum())
        tn = int(valid.sum() - tp - fp - fn)
        row = {
            "id": key, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "dice": ratio(2 * tp, 2 * tp + fp + fn, True),
            "iou": ratio(tp, tp + fp + fn, True),
            "recall": ratio(tp, tp + fn, True),
            "precision": ratio(tp, tp + fp, not target.any()),
        }
        if detailed:
            row.update(mask_geometry(target, valid))
            row["boundary_f1"] = boundary_f1(pred, target, valid)
            row["failure"] = row["dice"] < CONFIG["failure_dice"]
            row["missed_entire_polyp"] = bool(target.any() and not pred.any())
        rows.append(row)
    return rows

@torch.inference_mode()
def evaluate(model, split, detailed=False):
    model.eval()
    rows = []
    for images, targets, valid, keys in loader(split):
        probs = model(images.to(DEVICE, non_blocking=True))["out"].float().sigmoid().cpu()
        rows.extend(metric_rows(probs[:, 0].numpy() >= CONFIG["probability_threshold"],
                                targets[:, 0].numpy(), valid[:, 0].numpy(), keys, detailed))
    return pd.DataFrame(rows)

def objective(logits, target, valid, mode):
    target, valid = target.float(), valid.float()
    logits = logits.float()
    probability = logits.sigmoid() * valid
    dimensions = (1, 2, 3)
    area = target.sum(dim=dimensions)
    intersection = (probability * target).sum(dim=dimensions)
    dice_loss = 1.0 - ((2 * intersection + 1) / (probability.sum(dim=dimensions) + area + 1))
    weight = valid
    if mode == "small_positive":
        small = area / valid.sum(dim=dimensions) <= REFERENCE_CUTS["area_fraction"][0]
        emphasis = target * small[:, None, None, None]
        weight = weight * (1 + (CONFIG["upweight_factor"] - 1) * emphasis)
    elif mode == "boundary":
        dilated = F.max_pool2d(target, 5, stride=1, padding=2)
        eroded = 1 - F.max_pool2d(F.pad(1 - target, (2, 2, 2, 2), value=1), 5, stride=1)
        weight = weight * (1 + (CONFIG["upweight_factor"] - 1) * (dilated - eroded))
    pixel_bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    bce = (pixel_bce * weight).sum(dim=dimensions) / weight.sum(dim=dimensions)
    return (0.5 * bce + 0.5 * dice_loss).mean()

def fit(name, mode):
    latest = RUN / "checkpoints" / f"{name}_last.pt"
    best = RUN / "checkpoints" / f"{name}_best.pt"
    model = build_model()
    model.load_state_dict(torch.load(INITIAL_PATH, map_location="cpu", weights_only=True))
    model.to(DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": CONFIG["backbone_learning_rate"]},
        {"params": model.classifier.parameters(), "lr": CONFIG["learning_rate"]},
    ], weight_decay=CONFIG["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=CONFIG["amp"])
    history, first_epoch = [], 1
    best_dice, best_epoch, best_model = -1.0, None, None
    identity = {"run_id": RUN_ID, "initial_sha256": INIT_SHA, "mode": mode}
    if RESUME and latest.exists():
        state = torch.load(latest, weights_only=True, map_location="cpu")
        if any(state[key] != value for key, value in identity.items()):
            raise RuntimeError(f"재개 설정 불일치: {latest}. 새 작업 경로를 사용하거나 RESUME=False로 재실행하세요.")
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        history, first_epoch = state["history"], state["epoch"] + 1
        best_dice, best_epoch, best_model = state["best_dice"], state["best_epoch"], state["best_model"]
        del state
        print(f"{name}: completed {first_epoch - 1} epoch(s), resuming.")
    try:
        for epoch in range(first_epoch, CONFIG["epochs"] + 1):
            seed_everything(CONFIG["seed"] + epoch)
            model.train()
            total_loss = 0.0
            for images, targets, valid, _ in loader("train", epoch):
                images, targets, valid = [value.to(DEVICE, non_blocking=True)
                                          for value in (images, targets, valid)]
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=CONFIG["amp"]):
                    logits = model(images)["out"]
                loss = objective(logits, targets, valid, mode)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"{name}, epoch {epoch}: non-finite loss")
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += float(loss.detach()) * len(images)

            selection = evaluate(model, "val_select")
            means = selection[["dice", "iou", "recall", "precision"]].mean()
            history.append({"epoch": epoch, "train_loss": total_loss / len(SPLIT_IDS["train"]),
                            **{f"val_{metric}": float(value) for metric, value in means.items()}})
            if means.dice > best_dice:
                best_dice, best_epoch = float(means.dice), epoch
                best_model = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            save_checkpoint(latest, {**identity, "epoch": epoch, "model": model.state_dict(),
                                     "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                                     "best_model": best_model, "best_dice": best_dice,
                                     "best_epoch": best_epoch, "history": history})
            save_csv(pd.DataFrame(history), f"{name}_history.csv")
            print(f"{name:12} epoch {epoch:02}/{CONFIG['epochs']}  "
                  f"loss={history[-1]['train_loss']:.4f}  val Dice={means.dice:.4f}", flush=True)
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError("GPU 메모리 부족. 런타임을 다시 시작하고 BATCH_SIZE=4 또는 2로 새 실험을 실행하세요. "
                           f"동일 설정으로 재개할 때 사용할 완료 epoch 체크포인트: {latest}") from error
    except (RuntimeError, OSError, FloatingPointError, KeyboardInterrupt) as error:
        print(f"훈련 중단. 같은 설정으로 다시 실행하면 마지막 완료 epoch부터 재개합니다: {latest}", flush=True)
        raise
    save_checkpoint(best, {**identity, "epoch": best_epoch, "val_dice": best_dice, "model": best_model})
    history = pd.DataFrame(history)
    save_csv(history, f"{name}_history.csv")
    return history

def discover_checkpoints():
    if CHECKPOINT_PATH and UPLOAD_CHECKPOINT:
        raise ValueError("CHECKPOINT_PATH와 UPLOAD_CHECKPOINT 중 하나만 사용하세요.")
    if CHECKPOINT_PATH:
        return [(Path(CHECKPOINT_PATH).expanduser().resolve(), "지정 체크포인트", True)]
    if UPLOAD_CHECKPOINT:
        if not IN_COLAB:
            raise ValueError("업로드 창은 Colab에서 사용할 수 있습니다.")
        from google.colab import files
        uploads = files.upload()
        models = [name for name in uploads if Path(name).suffix == ".pt"]
        if len(models) != 1:
            raise ValueError("기존 baseline_best.pt 하나를 선택하세요. run_setup.json은 선택 사항입니다.")
        model_digest = hashlib.sha256(uploads[models[0]]).hexdigest()[:16]
        folder = ROOT / "uploaded" / model_digest
        folder.mkdir(parents=True, exist_ok=True)
        model_name = Path(models[0]).name
        for name in [models[0], "run_setup.json", "split_manifest.csv"]:
            if name in uploads:
                (folder / Path(name).name).write_bytes(uploads[name])
        return [(folder / model_name, "업로드 체크포인트", True)]

    own = RUN / "checkpoints" / "baseline_best.pt"
    cached = [(own, "재사용 체크포인트", False)] if own.is_file() else []
    if SMOKE:
        return cached
    candidates = []
    folders = [Path("/content/polyp_audit/runs"), Path("/content/drive/MyDrive/polyp_audit/runs"),
               ROOT / "training"]
    if IN_COLAB:
        candidates.extend((path, "Colab 파일에 있는 체크포인트", False)
                          for path in sorted(Path("/content").glob("baseline_best*.pt")))
    persisted = sorted((ROOT / "uploaded").glob("*/*.pt"),
                       key=lambda path: path.stat().st_mtime, reverse=True)
    candidates.extend((path, "이전에 업로드한 체크포인트", False) for path in persisted)
    candidates.extend(cached)
    for folder in folders:
        paths = sorted(folder.glob("*/checkpoints/baseline_best.pt"),
                       key=lambda path: (path.stat().st_mtime, path.as_posix()), reverse=True)
        candidates.extend((path, "이전 실행 체크포인트", False) for path in paths if path != own)
    return candidates

def checkpoint_manifest(path):
    meta_folder = path.parent.parent if path.parent.name == "checkpoints" else path.parent
    manifest_path = meta_folder / "run_setup.json"
    if not manifest_path.is_file():
        return None, meta_folder
    return json.loads(manifest_path.read_text(encoding="utf-8")), meta_folder

def use_checkpoint(path, source):
    if not path.is_file():
        raise FileNotFoundError(f"체크포인트가 없습니다: {path}")
    if path.stat().st_size > 80 * 2**20:
        raise ValueError("baseline_best.pt를 선택하세요. 학습 재시작용 전체 체크포인트와 결과 ZIP은 별도입니다.")
    manifest, folder = checkpoint_manifest(path)
    record = torch.load(path, weights_only=True, map_location="cpu")
    if not isinstance(record, dict) or record.get("mode") != "baseline" or "model" not in record:
        raise ValueError("기존 연구 노트북의 baseline_best.pt 형식을 기대합니다.")
    if manifest is not None:
        cfg = manifest["config"]
        if record["run_id"] != manifest["run_id"] or any(
            cfg[name] != CONFIG[name] for name in ("side", "profile", "seed", "mean", "std")
        ):
            raise ValueError("동봉된 실행 설정의 모델 ID, 해상도, 프로파일, 시드 또는 정규화가 일치하지 않습니다.")
        if manifest["split_sha256"] != SPLIT_SHA:
            columns = ["id", "group_id", "split"]
            old_manifest = folder / "split_manifest.csv"
            same_assignment = (old_manifest.is_file() and
                               pd.read_csv(old_manifest)[columns].equals(SPLIT[columns]))
            if not same_assignment:
                raise ValueError("학습 때 사용한 분할과 현재 분할이 일치하지 않습니다.")
    model = build_model()
    model.load_state_dict(record["model"], strict=True)
    with torch.inference_mode():
        model = model.to(DEVICE).eval()
        side = CONFIG["side"]
        output = model(torch.zeros((1, 3, side, side), device=DEVICE))["out"]
        if output.shape != (1, 1, side, side) or not torch.isfinite(output).all():
            raise ValueError("모델 출력의 모양이나 값이 올바르지 않습니다.")
    return model, {
        "source": source, "file": path.name, "epoch": int(record["epoch"]),
        "model_run_id": str(record["run_id"]), "sha256": sha256_file(path),
        "split_metadata_verified": manifest is not None,
    }

def prepare_model():
    for path, source, explicit in discover_checkpoints():
        try:
            return use_checkpoint(path, source)
        except (OSError, RuntimeError, ValueError, TypeError, KeyError, pickle.UnpicklingError) as error:
            if explicit:
                raise RuntimeError(f"선택한 체크포인트를 불러오지 못했습니다: {error}") from error
            print(f"체크포인트 건너뜀 · {path.name} · {error}")
    if DEVICE.type != "cuda" and not SMOKE:
        raise RuntimeError("Colab GPU를 선택하거나 baseline_best.pt를 업로드하세요. CPU 동작 확인은 PROFILE='smoke'입니다.")
    global INITIAL_PATH, INIT_SHA
    INITIAL_PATH = initialization()
    INIT_SHA = sha256_file(INITIAL_PATH)
    try:
        fit("baseline", "baseline")
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError("GPU 메모리가 부족합니다. BATCH_SIZE를 4 또는 2로 줄여 다시 실행하세요.") from error
    return use_checkpoint(RUN / "checkpoints" / "baseline_best.pt",
                          "스모크 테스트용 학습" if SMOKE else "이번 실행에서 기준 모델 학습")


BASELINE, MODEL_INFO = prepare_model()
print(f"{MODEL_INFO['source']} · {MODEL_INFO['file']} · 선택 epoch {MODEL_INFO['epoch']}")
if not MODEL_INFO["split_metadata_verified"]:
    print("가중치 구조는 확인했습니다. 원본 run_setup.json 없이 학습 분할의 일치는 확인할 수 없습니다.")

## 04 · 검수 후보와 우선순위

val_audit의 영상 경로와 모델 추론을 사용합니다. 네 방향 TTA 예측을 평균한 뒤 엔트로피 상위 5% 평균으로 검수 순위를 정하고, 후보 마스크를 원본 좌표에 복원합니다.

In [ ]:
class ReviewImages(Dataset):
    """Inference data source with image paths and no reference mask paths."""
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        key, path = self.items[index]
        with Image.open(path) as image:
            image = image.convert("RGB")
            rgb, _, valid = letterbox(image, Image.new("L", image.size), CONFIG["side"])
        normalized = (rgb.astype(np.float32) / 255.0 - MEAN) / STD
        return torch.from_numpy(normalized.transpose(2, 0, 1).copy()), torch.from_numpy(valid), key

@torch.inference_mode()
def infer_review_maps(model, image_items):
    model.eval()
    review_loader = DataLoader(ReviewImages(image_items), batch_size=CONFIG["batch_size"], shuffle=False)
    maps = {}
    for images, valid, keys in review_loader:
        images = images.to(DEVICE)
        tta_probs = []
        for axes in CONFIG["tta_flips"]:
            batch = images.flip(axes) if axes else images
            prob = model(batch)["out"].float().sigmoid()
            tta_probs.append(prob.flip(axes) if axes else prob)
        prob = torch.stack(tta_probs).mean(dim=0)
        safe = prob.clamp(1e-6, 1 - 1e-6)
        entropy = -(safe * safe.log2() + (1 - safe) * (1 - safe).log2())
        for key, probability, uncertainty, valid_mask in zip(
            keys, prob[:, 0].cpu().numpy(), entropy[:, 0].cpu().numpy(), valid.numpy(), strict=True
        ):
            maps[key] = {"p": probability, "entropy": uncertainty, "valid": valid_mask}
    return maps

def rank_predictions(maps):
    rows = []
    for key, values in maps.items():
        scores = values["entropy"][values["valid"]]
        k = max(1, math.ceil(CONFIG["entropy_top_fraction"] * scores.size))
        uncertainty = float(np.partition(scores, -k)[-k:].mean())
        rows.append({"id": key, "uncertainty_score": uncertainty})
    ranking = pd.DataFrame(rows).sort_values(["uncertainty_score", "id"], ascending=[False, True])
    return ranking.assign(review_rank=np.arange(1, len(ranking) + 1))

# Any version change to inference or case ordering starts a new review session.
signature = {"protocol": CONFIG["protocol"], "profile": PROFILE, "side": CONFIG["side"],
             "threshold": CONFIG["probability_threshold"], "tta": CONFIG["tta_flips"],
             "entropy_top_fraction": CONFIG["entropy_top_fraction"],
             "split": SPLIT_SHA, "model": MODEL_INFO["sha256"],
             "archive": DATA_INFO["archive_sha256"]}
SESSION_ID = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:12]
SESSION = ROOT / "sessions" / f"{PROFILE}_{SESSION_ID}"
SESSION.mkdir(parents=True, exist_ok=True)
AUDIT_IMAGES = [(key, PAIR_BY_ID[key][1]) for key in SPLIT_IDS["val_audit"]]

def prep_review_maps(model, image_items):
    path = SESSION / "inference_maps.npz"
    keys = [key for key, _ in image_items]
    if path.is_file():
        try:
            with np.load(path, allow_pickle=False) as arrays:
                if arrays["ids"].tolist() != keys:
                    raise ValueError("different case IDs")
                stacked = {name: arrays[name] for name in ("p", "entropy", "valid")}
            shape = (len(keys), CONFIG["side"], CONFIG["side"])
            if any(array.shape != shape or not np.isfinite(array).all() for array in stacked.values()):
                raise ValueError("invalid inference arrays")
            if stacked["valid"].dtype != bool or not stacked["valid"].any(axis=(1, 2)).all():
                raise ValueError("invalid image footprint")
            return {key: {name: array[i] for name, array in stacked.items()}
                    for i, key in enumerate(keys)}
        except (OSError, ValueError, KeyError, zipfile.BadZipFile) as error:
            if (SESSION / "review.sqlite3").exists():
                raise RuntimeError("검수에 사용한 추론 캐시가 손상됐습니다. 기존 세션 폴더를 백업하고 새 작업 경로로 실행하세요.") from error
            print(f"추론 캐시를 다시 생성합니다: {error}")
    maps = infer_review_maps(model, image_items)
    stacked = {name: np.stack([maps[key][name] for key in keys])
               for name in ("p", "entropy", "valid")}
    with tempfile.NamedTemporaryFile(dir=SESSION, suffix=".npz", delete=False) as temporary:
        temp_name = Path(temporary.name)
        try:
            np.savez_compressed(temporary, ids=np.asarray(keys), **stacked)
            temporary.flush()
            temp_name.replace(path)
        finally:
            temp_name.unlink(missing_ok=True)
    return maps

REVIEW_MAPS = prep_review_maps(BASELINE, AUDIT_IMAGES)
RANKING = rank_predictions(REVIEW_MAPS)
print(f"검수 세션 {SESSION_ID} · 후보 {len(RANKING)}장 · {SESSION}")

## 05 · 합성 주석 오류 실험

이미 탐색한 val_audit에 새로 만든 경미한 오류를 주입하는 오프라인 비교입니다. 모델·후보 마스크에서만 점수를 계산하고, 이후 주입 이력을 대조합니다. 반복과 평가 예산은 설정에 고정합니다.

In [ ]:
def score_candidate(candidate, maps):
    """Review score uses only the candidate annotation and image-derived model maps."""
    candidate = candidate & maps["valid"]
    probability, entropy, valid = maps["p"], maps["entropy"], maps["valid"]
    predicted = (probability >= CONFIG["probability_threshold"]) & valid
    support = (candidate | predicted) & valid
    if not support.any():
        return {"disagreement_score": 0.0, "annotation_review_score": 0.0}
    disagreement = candidate ^ predicted
    confidence = 2 * np.abs(probability - 0.5)
    # Entropy enters only on disagreeing pixels. The denominator excludes easy distant background.
    weighted = disagreement * (0.75 * confidence + 0.25 * entropy)
    return {"disagreement_score": float(disagreement.sum() / support.sum()),
            "annotation_review_score": float(weighted[support].mean())}

METHODS = {"random": "Random", "legacy": "Legacy", "contour": "Contour + legacy"}
ERROR_TYPES = CONFIG["injection_types"]

def contour_review_score(candidate, maps, legacy):
    """Compare proposal and model contours in the unpadded input grid."""
    valid = maps["valid"]
    predicted = (maps["p"] >= CONFIG["probability_threshold"]) & valid
    candidate = candidate & valid
    contour_a = candidate & ~ndimage.binary_erosion(candidate)
    contour_b = predicted & ~ndimage.binary_erosion(predicted)
    if not contour_a.any() or not contour_b.any():
        distance = float(contour_a.any() != contour_b.any())
    else:
        cap = CONFIG["contour_max_distance"]
        directions = [ndimage.distance_transform_edt(~contour_b)[contour_a],
                      ndimage.distance_transform_edt(~contour_a)[contour_b]]
        tails = []
        for distances in directions:
            count = max(1, math.ceil(distances.size * CONFIG["contour_top_fraction"]))
            tail = np.partition(distances, -count)[-count:]
            tails.append(float(np.minimum(tail / cap, 1).mean()))
        distance = float(np.mean(tails))
    weight = CONFIG["contour_weight"]
    return weight * distance + (1 - weight) * legacy

def corrupt_candidate(reference, valid, kind, rng):
    area = int(reference.sum())
    if kind == "clean" or not area:
        return reference.copy()
    radius = math.sqrt(area / math.pi)
    # Small shifts and morphology are defined in model-space pixels.
    if kind == "shift":
        magnitude = int(np.clip(round(radius * rng.uniform(0.04, 0.10)), 1, 5))
        delta = [0, 0]
        delta[int(rng.integers(2))] = magnitude * int(rng.choice([-1, 1]))
        result = ndimage.shift(reference, delta, order=0, mode="constant", cval=0, prefilter=False)
    elif kind in {"erosion", "dilation"}:
        distance = int(np.clip(round(radius * rng.uniform(0.025, 0.065)), 1, 3))
        transform = ndimage.binary_erosion if kind == "erosion" else ndimage.binary_dilation
        result = transform(reference, iterations=distance)
    elif kind == "local_missing":
        coordinates = np.argwhere(reference & ~ndimage.binary_erosion(reference))
        y, x = coordinates[int(rng.integers(len(coordinates)))]
        yy, xx = np.ogrid[:reference.shape[0], :reference.shape[1]]
        radius = max(1, round(radius * rng.uniform(0.20, 0.38)))
        result = reference & ((yy - y) ** 2 + (xx - x) ** 2 > radius ** 2)
    else:
        raise ValueError(f"Unknown injection {kind}")
    return result & valid

def geometry_guard(candidate, reference):
    """Keep a nonempty proposal and bound changed pixels to the mild-error regime."""
    changed = int((candidate != reference).sum())
    area = int(reference.sum())
    if candidate.any() and 0 < changed <= max(1, int(0.35 * area)):
        return candidate
    return reference.copy()

def atomic_csv(frame, path):
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix=".part", delete=False) as temporary:
        name = Path(temporary.name)
    try:
        frame.to_csv(name, index=False)
        name.replace(path)
    finally:
        name.unlink(missing_ok=True)

def run_injections():
    directory = SESSION / "offline_experiment"
    directory.mkdir(exist_ok=True)
    keys = SPLIT_IDS["val_audit"]
    references = {key: load_pair(*PAIR_BY_ID[key])[1] for key in keys}
    scores, truth = [], []
    for trial in range(1, CONFIG["injection_repeats"] + 1):
        rng = np.random.default_rng([CONFIG["seed"], 1100, trial])
        shuffled_ids = rng.permutation(keys)
        shuffled_types = rng.permutation(ERROR_TYPES)
        changed_count = int(round(len(keys) * CONFIG["injection_fraction"]))
        for index, key in enumerate(shuffled_ids):
            key = str(key)
            kind = str(shuffled_types[index % len(shuffled_types)]) if index < changed_count else "clean"
            reference, model_maps = references[key], REVIEW_MAPS[key]
            candidate = geometry_guard(corrupt_candidate(reference, model_maps["valid"], kind, rng), reference)
            legacy = score_candidate(candidate, model_maps)["annotation_review_score"]
            scores.append({"trial": trial, "id": key, "legacy": legacy,
                           "contour": contour_review_score(candidate, model_maps, legacy)})
            changed = int((candidate != reference).sum())
            union = int((candidate | reference).sum())
            truth.append({"trial": trial, "id": key, "requested_error": kind,
                          "is_error": changed > 0, "changed_pixels": changed,
                          "reference_iou": float((candidate & reference).sum() / union) if union else 1.0})
    blind = pd.DataFrame(scores)
    for name in ("legacy", "contour"):
        ordered = blind.sort_values(["trial", name, "id"], ascending=[True, False, True])
        blind[f"{name}_rank"] = ordered.groupby("trial").cumcount().add(1)
    atomic_csv(blind, directory / "synthetic_blind_scores.csv")
    cases = blind.merge(pd.DataFrame(truth), on=["trial", "id"], validate="one_to_one")
    atomic_csv(cases, directory / "synthetic_cases.csv")

    rows = []
    for trial, frame in cases.groupby("trial", sort=True):
        frame = frame.sort_values("id").reset_index(drop=True)
        rng = np.random.default_rng([CONFIG["seed"], 1300, int(trial)])
        random_orders = np.stack([rng.permutation(len(frame)) for _ in range(CONFIG["random_repeats"])])
        for budget in CONFIG["review_budgets"]:
            k = math.ceil(len(frame) * budget)
            for kind in ["all", *ERROR_TYPES]:
                positive = frame.is_error.to_numpy(dtype=bool, copy=True)
                if kind != "all":
                    positive &= frame.requested_error.eq(kind).to_numpy()
                n_positive = int(positive.sum())
                random_found = positive[random_orders[:, :k]].sum(axis=1)
                for method, found in (
                    ("random", float(random_found.mean())),
                    ("legacy", int(positive[frame.legacy_rank.le(k)].sum())),
                    ("contour", int(positive[frame.contour_rank.le(k)].sum())),
                ):
                    rows.append({"trial": int(trial), "budget": budget, "reviewed": k,
                                 "error_type": kind, "method": method, "n_errors": n_positive,
                                 "found": found, "capture": found / n_positive if n_positive else np.nan})
    comparisons = pd.DataFrame(rows)
    atomic_csv(comparisons, directory / "synthetic_budgets_by_trial.csv")
    summary = (comparisons.groupby(["budget", "error_type", "method"], as_index=False)
               .agg(reviewed=("reviewed", "first"), trials=("trial", "size"),
                    mean_found=("found", "mean"), total_errors=("n_errors", "sum"),
                    total_found=("found", "sum")))
    summary["capture"] = summary.total_found / summary.total_errors.replace(0, np.nan)
    atomic_csv(summary, directory / "synthetic_summary.csv")
    plot_synthetic(summary, directory)
    return summary, cases

def plot_synthetic(summary, directory):
    colors = {"random": "#697384", "legacy": "#2779A7", "contour": "#007F73"}
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.3), layout="constrained")
    positions = np.arange(len(ERROR_TYPES))
    at_twenty = summary[summary.budget.eq(0.2)]
    for index, (method, label) in enumerate(METHODS.items()):
        all_types = summary[summary.error_type.eq("all") & summary.method.eq(method)]
        left.plot(100 * all_types.budget, 100 * all_types.capture, "o-", color=colors[method], label=label)
        by_type = at_twenty[at_twenty.method.eq(method)].set_index("error_type").reindex(ERROR_TYPES)
        right.bar(positions + (index - 1) * 0.25, 100 * by_type.capture, width=0.23, color=colors[method])
    left.set(title="Synthetic errors found by review budget", xlabel="Review budget (%)",
             ylabel="Error capture (%)", xticks=[10, 20, 30], ylim=(0, 105))
    right.set(title="Error types at a 20% review budget", ylabel="Error capture (%)",
              xticks=positions, xticklabels=["Shift", "Erosion", "Dilation", "Local miss"], ylim=(0, 105))
    left.legend(frameon=False, loc="upper left")
    fig.savefig(directory / "synthetic_annotation_comparison.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

EXPERIMENT_SUMMARY, EXPERIMENT_CASES = run_injections()
display(EXPERIMENT_SUMMARY[EXPERIMENT_SUMMARY.error_type.eq("all")].round(3))

## 06 · 검수 기록과 결과 내보내기

결정과 이진 마스크를 함께 기록합니다. 런타임을 교체해도 기록을 유지하려면 첫 실행부터 `MOUNT_DRIVE=True`를 설정하고 같은 세션을 이어갑니다.

In [ ]:
from contextlib import closing
from functools import lru_cache
import re

MASK_RGB = (28, 192, 159)
MASK_HEX = "#1cc09f"
STATUSES = {"approved": "승인", "corrected": "수정 완료", "deferred": "보류"}
LOG_COLUMNS = [
    "case_id", "rank", "uncertainty_score", "status", "reason", "elapsed_seconds",
    "decided_at_utc", "before_mask", "after_mask", "changed_pixels", "actor",
]

@dataclass(frozen=True)
class Case:
    key: str
    rank: int
    score: float
    rgb: np.ndarray
    candidate: np.ndarray
    entropy: np.ndarray

def lift_from_letterbox(array, valid, output_size, resample):
    ys, xs = np.nonzero(valid)
    region = array[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    return np.asarray(Image.fromarray(region).resize(output_size, resample))

class ReviewQueue:
    """The application receives paths to images and arrays from model inference only."""
    def __init__(self, ranking, items, maps):
        self.table = {row.id: (int(row.review_rank), float(row.uncertainty_score))
                      for row in ranking.itertuples(index=False)}
        if any(not re.fullmatch(r"[A-Za-z0-9_-]+", key) for key in self.table):
            raise ValueError("지원하지 않는 이미지 ID입니다.")
        self.keys = tuple(self.table)
        self.images = dict(items)
        self.maps = maps

    @lru_cache(maxsize=4)
    def get(self, key):
        with Image.open(self.images[key]) as image:
            size, rgb = image.size, np.asarray(image.convert("RGB"))
        maps = self.maps[key]
        model_mask = (maps["p"] >= CONFIG["probability_threshold"]) & maps["valid"]
        candidate = lift_from_letterbox(model_mask.astype(np.uint8), maps["valid"],
                                        size, Image.Resampling.NEAREST).astype(bool)
        entropy = lift_from_letterbox(maps["entropy"], maps["valid"], size, Image.Resampling.BILINEAR)
        rank, score = self.table[key]
        return Case(key, rank, score, rgb, candidate, entropy)

def overlay_mask(case, mask, opacity):
    source = case.rgb.copy()
    source[mask] = (source[mask] * (1 - opacity) + np.asarray(MASK_RGB) * opacity).astype(np.uint8)
    return source

def heatmap(case):
    color = plt.colormaps["magma"](np.clip(case.entropy, 0, 1), bytes=True)[..., :3]
    return color

def editor_payload(case):
    mask = np.zeros((*case.candidate.shape, 4), dtype=np.uint8)
    mask[case.candidate] = (*MASK_RGB, 255)
    return {"background": Image.fromarray(case.rgb).convert("RGBA"),
            "layers": [Image.fromarray(mask)],
            "composite": Image.fromarray(overlay_mask(case, case.candidate, 1)).convert("RGBA")}

def decode_editor(case, payload):
    if not isinstance(payload, dict) or len(payload.get("layers", [])) != 1:
        raise ValueError("마스크 레이어를 다시 열고 수정해 주세요.")
    layer = np.asarray(payload["layers"][0])
    if layer.shape != (*case.candidate.shape, 4):
        raise ValueError("편집 캔버스 크기가 원본과 다릅니다. 수정 화면을 다시 열어 주세요.")
    return layer[..., 3] >= 128

def png_mask(mask):
    with io.BytesIO() as buffer:
        Image.fromarray(mask.astype(np.uint8) * 255).save(buffer, format="PNG")
        return buffer.getvalue()

def write_bytes_atomic(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix=".part", delete=False) as temporary:
        name = Path(temporary.name)
        try:
            temporary.write(content)
            temporary.flush()
            os.fsync(temporary.fileno())
            name.replace(path)
        finally:
            name.unlink(missing_ok=True)

class ReviewStore:
    def __init__(self, directory, session_id, queue, actor):
        self.directory, self.session_id, self.queue, self.actor = directory, session_id, queue, actor
        self.lock = RLock()
        self.db = directory / "review.sqlite3"
        self.directory.mkdir(parents=True, exist_ok=True)
        with closing(self.connect()) as conn, conn:
            conn.execute("CREATE TABLE IF NOT EXISTS session (id TEXT PRIMARY KEY)")
            prior = conn.execute("SELECT id FROM session").fetchall()
            if prior and [row["id"] for row in prior] != [session_id]:
                raise ValueError("이 검수 기록의 세션이 현재 영상·모델과 다릅니다.")
            conn.execute("INSERT OR IGNORE INTO session VALUES (?)", (session_id,))
            conn.execute("""
                CREATE TABLE IF NOT EXISTS reviews (
                    case_id TEXT PRIMARY KEY, rank INTEGER UNIQUE NOT NULL,
                    uncertainty_score REAL NOT NULL, status TEXT NOT NULL,
                    reason TEXT NOT NULL, elapsed_seconds REAL NOT NULL,
                    decided_at_utc TEXT NOT NULL, before_mask TEXT NOT NULL,
                    after_mask TEXT NOT NULL, changed_pixels INTEGER, actor TEXT NOT NULL,
                    before_png BLOB NOT NULL, after_png BLOB
                )
            """)
        self.sync_files()

    def connect(self):
        conn = sqlite3.connect(self.db, timeout=30)
        conn.row_factory = sqlite3.Row
        return conn

    def pending(self, conn=None):
        if conn is None:
            with closing(self.connect()) as db:
                return self.pending(db)
        completed = {row["case_id"] for row in conn.execute("SELECT case_id FROM reviews")}
        return next((key for key in self.queue.keys if key not in completed), None)

    def rows(self):
        with closing(self.connect()) as conn:
            return [dict(row) for row in conn.execute("SELECT * FROM reviews ORDER BY rank")]

    def commit(self, key, status, reason, elapsed, mask=None):
        if status not in STATUSES:
            raise ValueError("지원하지 않는 검수 상태입니다.")
        with self.lock, closing(self.connect()) as conn, conn:
            conn.execute("BEGIN IMMEDIATE")
            if key is None or self.pending(conn) != key:
                return False
            case = self.queue.get(key)
            if status == "corrected":
                if mask is None or mask.dtype != bool or mask.shape != case.candidate.shape:
                    raise ValueError("원본 크기의 이진 마스크가 필요합니다.")
                if not (mask != case.candidate).any():
                    raise ValueError("수정된 픽셀이 없습니다. 승인 버튼으로 후보를 확정할 수 있습니다.")
            if status == "approved":
                mask = case.candidate
            if status == "deferred":
                mask = None
            before = f"masks/before/{key}.png"
            after = f"masks/after/{key}.png" if mask is not None else ""
            row = [key, case.rank, case.score, status, str(reason or "").strip()[:400],
                   round(max(0.0, elapsed), 3), datetime.now(timezone.utc).isoformat(timespec="seconds"),
                   before, after, int((mask != case.candidate).sum()) if mask is not None else None,
                   self.actor, png_mask(case.candidate), png_mask(mask) if mask is not None else None]
            conn.execute("INSERT INTO reviews VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", row)
        try:
            self.sync_files()
        except OSError as error:
            warnings.warn(f"검수 판단은 DB에 저장했습니다. PNG·CSV는 폴더 복구 또는 내보내기에서 다시 생성합니다: {error}")
        return True

    def sync_files(self):
        with self.lock:
            rows = self.rows()
            for row in rows:
                for relative, blob in ((row["before_mask"], row["before_png"]), (row["after_mask"], row["after_png"])):
                    if relative:
                        path = self.directory / relative
                        if not path.is_file() or path.read_bytes() != blob:
                            write_bytes_atomic(path, blob)
            frame = pd.DataFrame([{name: row[name] for name in LOG_COLUMNS} for row in rows],
                                 columns=LOG_COLUMNS)
            atomic_csv(frame, self.directory / "review_log.csv")
            return rows

def export_report(summary, rows):
    actual = [row for row in rows if row["actor"] == "human"]
    lines = [
        "# 내시경 용종 마스크 검수 결과",
        "",
        f"검수 세션 `{SESSION_ID}` · 후보 {len(RANKING)}장 · {ENVIRONMENT['gpu'] or DEVICE.type}.",
        f"모델 {MODEL_INFO['source']} · `{MODEL_INFO['file']}` · epoch {MODEL_INFO['epoch']}.",
        f"가중치와 분할 정보 일치 확인 {MODEL_INFO['split_metadata_verified']}.",
        f"실행 유형 {'합성 데이터 개발 점검' if SMOKE else 'Kvasir-SEG 공개 데이터 연구'}.",
        "",
        "## 사람 검수 기록",
        "",
        f"사람 검수 **{len(actual)}건**."
    ]
    for status, label in STATUSES.items():
        lines.append(f"- {label} {sum(row['status'] == status for row in actual)}건")
    if actual:
        lines.extend(["", f"화면 표시부터 서버에 판단을 제출하기까지 걸린 시간의 중앙값은 "
                            f"{np.median([row['elapsed_seconds'] for row in actual]):.1f}초입니다. "
                            "자리 비움과 화면 전송 시간도 포함될 수 있습니다."])
    if len(rows) != len(actual):
        lines.extend(["", f"개발 점검 기록 {len(rows) - len(actual)}건은 사람 검수 집계에서 제외했습니다."])
    lines.extend([
        "", "## 합성 주석 오류 오프라인 평가", "",
        "같은 후보 목록과 검수 예산에 세 순위를 적용했습니다. 검수 예산별 수치는 반복 평균이며, 포착률은 실제 변경 사례의 누적 비율입니다.",
        "", "| 검수 예산 | 방식 | 평균 발견 건수 | 오류 포착률 |",
        "| --- | --- | --- | --- |",
    ])
    method_name = {"random": "무작위", "legacy": "기존 점수", "contour": "새 경계 점수"}
    for row in summary[summary.error_type.eq("all")].sort_values(["budget", "method"]).itertuples():
        capture = f"{row.capture:.1%}" if pd.notna(row.capture) else "계산 불가"
        lines.append(f"| {row.budget:.0%} | {method_name[row.method]} | {row.mean_found:.2f} | {capture} |")
    lines.extend([
        "", "![합성 오류 비교](offline_experiment/synthetic_annotation_comparison.png)", "",
        "기존 연구의 val_audit를 재사용한 탐색 평가입니다. 반복 간에 이미지가 재사용됩니다. "
        "참조 마스크와 주입 이력은 합성 후보 생성과 점수 산출 이후의 평가에 쓰였습니다.",
        "", "검수 화면에는 모델이 제안한 마스크가 표시됩니다. 수동 판단은 검수 기록으로 해석합니다. "
        "시간 절감 효과나 임상 성능은 이 실행에서 측정하지 않았습니다.",
        "", f"Jha et al. (2020). Kvasir-SEG. [공식 배포]({OFFICIAL_PAGE}) · [논문]({PAPER})", "",
    ])
    return "\n".join(lines)

def export_review(store):
    with store.lock:
        rows = store.sync_files()
        report = store.directory / "REVIEW_REPORT.md"
        write_bytes_atomic(report, export_report(EXPERIMENT_SUMMARY, rows).encode("utf-8"))
        output = store.directory / "review_results.zip"
        fixed = ["REVIEW_REPORT.md", "review_log.csv", "review_queue.csv", "review_setup.json"]
        paths = fixed + [name for row in rows for name in (row["before_mask"], row["after_mask"]) if name]
        paths.extend("offline_experiment/" + path.name
                     for path in sorted((store.directory / "offline_experiment").iterdir())
                     if path.suffix in {".csv", ".png"})
        with tempfile.NamedTemporaryFile(dir=store.directory, suffix=".zip.part", delete=False) as temporary:
            name = Path(temporary.name)
        try:
            with zipfile.ZipFile(name, "w", compression=zipfile.ZIP_DEFLATED) as archive:
                for relative in paths:
                    archive.write(store.directory / relative, relative)
            with zipfile.ZipFile(name) as archive:
                if archive.testzip() is not None:
                    raise RuntimeError("결과 ZIP 검증에 실패했습니다. 내보내기를 다시 실행하세요.")
            name.replace(output)
        finally:
            name.unlink(missing_ok=True)
    return output

QUEUE = ReviewQueue(RANKING, AUDIT_IMAGES, REVIEW_MAPS)
STORE = ReviewStore(SESSION, SESSION_ID, QUEUE, "smoke_test" if SMOKE else "human")
atomic_csv(RANKING, SESSION / "review_queue.csv")
write_json(SESSION / "review_setup.json", {
    "session_id": SESSION_ID, "model": MODEL_INFO, "environment": ENVIRONMENT, "config": CONFIG,
    "dataset": DATA_INFO, "split_sha256": SPLIT_SHA,
    "training_split": "train; checkpoint selected on val_select",
    "review_and_synthetic_split": "val_audit; fixed exploratory comparison using the previously examined split",
    "candidate_origin": "four-flip mean probability >= 0.50, inverse letterbox nearest neighbor",
    "score_tuning": "No score selected or fitted on this run. Parameters fixed in settings.",
    "time_measurement": "Wall elapsed time between showing a case and decision. Idle time may be included.",
})

## 07 · 검수 화면 구성

마스크 표시를 끄거나 투명도를 조정할 수 있습니다. 수정할 때에는 브러시·지우개를 사용한 뒤 반드시 **수정 미리보기 → 수정본 저장**을 누릅니다.

In [ ]:
class ReviewActions:
    def __init__(self, store):
        self.store = store
        self.offers = {}
        self.lock = RLock()

    def make_offer(self, key):
        with self.lock:
            token = secrets.token_urlsafe(18)
            self.offers[token] = (key, time.monotonic())
            if len(self.offers) > 128:
                self.offers.pop(next(iter(self.offers)))
            return token

    def active(self, token):
        with self.lock:
            entry = self.offers.get(token)
            if entry is None or self.store.pending() != entry[0]:
                raise ValueError("이미 처리된 사례이거나 화면이 오래됐습니다. 새로고침으로 검수를 이어가세요.")
            return self.store.queue.get(entry[0]), entry[1]

    def preview(self, token, payload):
        case, _ = self.active(token)
        mask = decode_editor(case, payload)
        changed = int((mask != case.candidate).sum())
        if changed == 0:
            raise ValueError("변경된 픽셀이 없습니다. 마스크를 편집한 뒤 미리보기를 누르세요.")
        return case, mask, changed, self.edit_digest(token, mask)

    @staticmethod
    def edit_digest(token, mask):
        return hashlib.sha256(token.encode() + mask.tobytes()).hexdigest()

    def commit(self, token, status, reason, payload=None, digest=None):
        case, opened_at = self.active(token)
        elapsed = time.monotonic() - opened_at
        if elapsed < 0.55:
            raise ValueError("다음 사례가 표시되었습니다. 현재 영상을 확인한 뒤 선택해 주세요.")
        edited = None
        if status == "corrected":
            _, edited, _, expected = self.preview(token, payload)
            if digest != expected:
                raise ValueError("현재 수정 상태를 저장하기 전에 수정 미리보기를 다시 확인해 주세요.")
        if not self.store.commit(case.key, status, reason, elapsed, edited):
            raise ValueError("이미 처리된 사례입니다. 화면을 새로고침하세요.")
        with self.lock:
            self.offers.pop(token, None)
        return case

UI_CSS = """
.gradio-container { max-width: 1280px !important; }
#main-heading h1 { font-size: clamp(1.55rem, 3vw, 2.4rem); margin-bottom: .2rem; }
#case-heading { border-left: 3px solid #128674; padding-left: 12px; }
#top-images img { object-fit: contain; }
#action-feedback { min-height: 2.15rem; }
"""

def build_review_app(store):
    actions = ReviewActions(store)
    with gr.Blocks(title="내시경 용종 마스크 검수") as app:
        gr.Markdown("# 내시경 용종 마스크 검수", elem_id="main-heading")
        model_caption = f"가중치 **{MODEL_INFO['source']}** · epoch **{MODEL_INFO['epoch']}**"
        if not MODEL_INFO["split_metadata_verified"]:
            model_caption += " · 체크포인트의 학습 분할 미확인"
        if SMOKE:
            model_caption += " · **합성 데이터 개발 점검**"
        gr.Markdown(model_caption)
        case_heading = gr.Markdown(elem_id="case-heading")
        with gr.Row(elem_id="top-images"):
            original = gr.Image(label="원본 영상", interactive=False, buttons=["fullscreen"], height=275, format="png")
            overlay = gr.Image(label="후보 마스크", interactive=False, buttons=["fullscreen"], height=275, format="png")
            heat = gr.Image(label="불확실성 히트맵", interactive=False, buttons=["fullscreen"], height=275, format="png")
        with gr.Row():
            show_mask = gr.Checkbox(value=True, label="마스크 표시", scale=1)
            opacity = gr.Slider(0, 1, value=0.45, step=0.05, label="마스크 투명도 조절", scale=3)
        note = gr.Textbox(label="검수 사유", placeholder="예: 경계 일부 누락, 반사광으로 판단 어려움",
                          lines=1, max_lines=3)
        with gr.Row():
            approve = gr.Button("승인", variant="primary")
            edit = gr.Button("수정")
            defer = gr.Button("보류")
        status = gr.Markdown(elem_id="action-feedback")
        with gr.Group(visible=False) as edit_group:
            gr.Markdown("브러시로 용종 영역을 추가하고 지우개로 지운 뒤 수정 미리보기를 확인하세요.")
            canvas = gr.ImageEditor(
                label="마스크 편집", type="numpy", image_mode="RGBA",
                sources=(), transforms=(), buttons=["fullscreen"], format="png", height=440,
                brush=gr.Brush(default_size=18, colors=[MASK_HEX], color_mode="fixed"),
                eraser=gr.Eraser(default_size=18),
                layers=gr.LayerOptions(allow_additional_layers=False, layers=["후보 마스크"]),
                interactive=True, elem_id="mask-editor",
            )
            with gr.Row():
                preview_button = gr.Button("수정 미리보기")
                save_edit = gr.Button("수정본 저장", variant="primary")
                cancel_edit = gr.Button("수정 취소")
            preview_image = gr.Image(label="수정본 미리보기", visible=False, interactive=False,
                                     buttons=["fullscreen"], height=300, format="png", elem_id="mask-preview")
        with gr.Row():
            export = gr.Button("결과 내보내기")
            file_output = gr.DownloadButton("결과 ZIP 다운로드", interactive=False,
                                            elem_id="download-review-zip")
        token = gr.Textbox(visible=False)
        prior_preview = gr.Textbox(visible=False)

        render_outputs = [case_heading, original, overlay, heat, token, note, edit_group, canvas,
                          preview_image, prior_preview, status, approve, edit, defer]

        def render(show, alpha, message=""):
            key = store.pending()
            completed = len(store.rows())
            total = len(store.queue.keys)
            if key is None:
                return ("### 검수 완료\n모든 후보를 검수했습니다. 결과를 내보낼 수 있습니다.",
                        None, None, None, "", "", gr.update(visible=False), gr.skip(),
                        gr.update(visible=False, value=None), "", message,
                        *[gr.update(interactive=False) for _ in range(3)])
            case = store.queue.get(key)
            caption = (f"### 사례 {case.rank} / {total}\n"
                       f"`{case.key}` · 불확실성 {case.score:.3f} · 판단 기록 {completed}건")
            return (caption, case.rgb, overlay_mask(case, case.candidate, alpha if show else 0),
                    heatmap(case), actions.make_offer(key), "", gr.update(visible=False), gr.skip(),
                    gr.update(visible=False, value=None), "", message,
                    *[gr.update(interactive=True) for _ in range(3)])

        def update_overlay(current_token, show, alpha):
            try:
                case, _ = actions.active(current_token)
                return overlay_mask(case, case.candidate, alpha if show else 0)
            except ValueError:
                return gr.skip()

        def open_editor(current_token):
            try:
                case, _ = actions.active(current_token)
                return (gr.update(visible=True), editor_payload(case),
                        gr.update(visible=False, value=None), "",
                        "수정 내용을 확인한 뒤 저장하세요.", gr.update(interactive=False))
            except ValueError as error:
                return (*[gr.skip() for _ in range(4)], str(error), gr.skip())

        def close_editor():
            return (gr.update(visible=False), gr.skip(), gr.update(visible=False, value=None), "",
                    "진행 중 편집을 닫았습니다.", gr.update(interactive=True))

        def make_preview(current_token, payload, alpha):
            try:
                case, updated, changed, digest = actions.preview(current_token, payload)
                return (gr.update(value=overlay_mask(case, updated, max(alpha, 0.45)), visible=True),
                        digest, f"수정된 픽셀 {changed:,}개. 확인 후 **수정본 저장**을 눌러 주세요.")
            except ValueError as error:
                return gr.update(value=None, visible=False), "", str(error)

        def save_action(next_status, current_token, reason, show, alpha, payload=None, digest=None):
            try:
                case = actions.commit(current_token, next_status, reason, payload, digest)
                return render(show, alpha, f"**사례 {case.rank}** · {STATUSES[next_status]} 기록을 저장했습니다.")
            except ValueError as error:
                outputs = [gr.skip() for _ in render_outputs]
                outputs[10] = str(error)
                return tuple(outputs)

        def make_archive():
            try:
                path = export_review(store)
                return gr.update(value=str(path), interactive=True), "결과 ZIP을 준비했습니다."
            except (OSError, ValueError, RuntimeError) as error:
                return gr.update(value=None, interactive=False), f"내보내기 실패 · {error}"

        options = {"api_name": False, "concurrency_id": "review", "concurrency_limit": 1}
        app.load(render, [show_mask, opacity], render_outputs, **options)
        show_mask.change(update_overlay, [token, show_mask, opacity], overlay, **options)
        opacity.release(update_overlay, [token, show_mask, opacity], overlay, **options)
        edit.click(open_editor, token, [edit_group, canvas, preview_image, prior_preview, status, approve], **options)
        cancel_edit.click(close_editor, None, [edit_group, canvas, preview_image, prior_preview, status, approve], **options)
        preview_button.click(make_preview, [token, canvas, opacity], [preview_image, prior_preview, status], **options)
        approve.click(lambda *args: save_action("approved", *args),
                      [token, note, show_mask, opacity], render_outputs, **options)
        defer.click(lambda *args: save_action("deferred", *args),
                    [token, note, show_mask, opacity], render_outputs, **options)
        save_edit.click(lambda *args: save_action("corrected", *args),
                        [token, note, show_mask, opacity, canvas, prior_preview], render_outputs, **options)
        export.click(make_archive, None, [file_output, status], **options)
    return app.queue(default_concurrency_limit=1)

## 08 · 검수 시작

출력에 표시된 주소를 열고 그 아래 사용자 이름과 임시 암호를 입력합니다. 일부만 검수한 상태에서도 **결과 내보내기 → 결과 ZIP 다운로드**를 사용할 수 있습니다.

In [ ]:
if "REVIEW_APP" in globals():
    REVIEW_APP.close()
REVIEW_APP = build_review_app(STORE)
options = {
    "server_name": "127.0.0.1", "prevent_thread_lock": True, "inline": False,
    "max_file_size": "20mb", "footer_links": [], "enable_monitoring": False,
}
if IN_COLAB:
    credentials = ("reviewer", secrets.token_urlsafe(16))
    options.update(share=True, auth=credentials, auth_message="이 노트북의 공개 데이터 검수 화면입니다.")
else:
    options.update(share=False)

review_theme = gr.themes.Soft(primary_hue="teal", neutral_hue="slate")
_, local_url, share_url = REVIEW_APP.launch(
    **options, theme=review_theme, css=UI_CSS,
    allowed_paths=[str(SESSION / "review_results.zip")],
    blocked_paths=[str(CACHE), str(ROOT / "training"), str(ROOT / "uploaded"),
                   str(SESSION / "review.sqlite3"), str(SESSION / "inference_maps.npz")],
)
if IN_COLAB:
    if not share_url:
        REVIEW_APP.close()
        raise RuntimeError("Colab 연결 링크를 만들지 못했습니다. 잠시 후 마지막 셀을 다시 실행하세요.")
    print(f"검수 화면: {share_url}\n사용자 이름: {credentials[0]}\n암호: {credentials[1]}")
    print("공개 Kvasir-SEG 연구 데이터용 세션입니다. 완료하면 Colab 런타임을 종료하세요.")
else:
    print(f"검수 화면: {local_url}")